# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import pandas as pd
pd.set_option('display.max_rows', 700)
pd.set_option('display.max_columns', 700)

In [2]:
import os
from src.config import Configuration


CONFIG = Configuration(

)

# Resize Tensors

In [3]:
import os
import torch
import torch.nn.functional as F

os.makedirs(CONFIG.mr_nf_tensors_96, exist_ok=True)

TARGET = 96

for fname in os.listdir(CONFIG.mr_nf_tensors):
    if not fname.endswith(".pt"):
        continue
    vol = torch.load(os.path.join(CONFIG.mr_nf_tensors, fname), weights_only=True)  # (4, H, W, D)
    vol = F.interpolate(
        vol.unsqueeze(0), size=(TARGET, TARGET, TARGET),
        mode="trilinear", align_corners=False
    ).squeeze(0)
    torch.save(vol, os.path.join(CONFIG.mr_nf_tensors_96, fname))
    print(f"saved {fname}  {vol.shape}")

# VIEW

In [4]:
import torch
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, IntSlider

def plot_patient(tensor_path):
    # Load stacked tensor (4, X, Y, Z)
    volume_tensor = torch.load(tensor_path, weights_only=True)
    volume_data = volume_tensor.numpy()
    
    modalities = ['T1', 'T1-Gd', 'T2', 'FLAIR']

    def _update_plot(slice_idx):
        # Create a 1x4 grid
        fig = make_subplots(
            rows=1, cols=4, 
            subplot_titles=modalities,
            horizontal_spacing=0.02
        )

        for i, mod_name in enumerate(modalities):
            img_slice = volume_data[i, :, :, slice_idx]
            
            fig.add_trace(
                go.Heatmap(
                    z=img_slice,
                    colorscale='Gray',
                    showscale=False,
                    transpose=True
                ),
                row=1, col=i+1
            )

        fig.update_layout(
            title_text=f"Patient: {os.path.basename(tensor_path)} | Axial Slice: {slice_idx}",
            width=1200, # Wide enough for 4 images
            height=400,
            margin=dict(l=10, r=10, t=50, b=10)
        )
        
        # Ensure consistent aspect ratios across all subplots
        fig.update_xaxes(visible=False)
        fig.update_yaxes(visible=False, scaleanchor="x", scaleratio=1)
        
        fig.show()

    # Slider for depth (Z-axis)
    interact(
        _update_plot, 
        slice_idx=IntSlider(
            min=0, 
            max=volume_data.shape[3]-1, 
            step=1, 
            value=volume_data.shape[3]//2,
            description='Slice Z:'
        )
    )

# Usage
# plot_all_modalities_plotly("data/tensors/UPENN-GBM-00001.pt")

In [7]:
file_path = os.path.join(CONFIG.mr_nf_tensors_96, "UPENN-GBM-00465.pt")
plot_patient(file_path)

interactive(children=(IntSlider(value=48, description='Slice Z:', max=95), Output()), _dom_classes=('widget-in…

In [ ]:
file_path = os.path.join(CONFIG.mr_nf_tensors_96, "UPENN-GBM-00001.pt")
plot_patient(file_path)

interactive(children=(IntSlider(value=77, description='Slice Z:', max=154), Output()), _dom_classes=('widget-i…

# How to clean tabular

In [6]:
import pandas as pd

df = pd.read_csv(CONFIG.mr_data)
# df = df[df['ID'].str.endswith('_11')].copy()

len(df)

671

In [7]:
# 1. Create a temporary column with the Patient Root ID (e.g., UPENN-GBM-00612)
df['PatientRoot'] = df['ID'].str.split('_').str[0]
# 2. Sort by ID so that _11 comes before _21 for the same patient
df = df.sort_values(by='ID')
# 3. Drop duplicates based on the Root ID, keeping the first one found
# This keeps _11 if it exists; otherwise, it keeps _21.
df = df.drop_duplicates(subset='PatientRoot', keep='first')
# 4. Clean up the temporary column
df = df.drop(columns=['PatientRoot'])

len(df)

630

In [8]:
# 1. Coerce invalid strings to NaN
df['Survival_from_surgery_days_UPDATED'] = pd.to_numeric(df['Survival_from_surgery_days_UPDATED'], errors='coerce')

# 2. Drop rows where survival is NaN
df = df.dropna(subset=['Survival_from_surgery_days_UPDATED'])

# 3. Now safe to convert to int
df['Survival_from_surgery_days_UPDATED'] = df['Survival_from_surgery_days_UPDATED'].astype(int)

df["Survival_from_surgery_days_UPDATED"].describe()

count     603.000000
mean      510.112769
std       518.107626
min         3.000000
25%       170.500000
50%       384.000000
75%       630.500000
max      6109.000000
Name: Survival_from_surgery_days_UPDATED, dtype: float64

In [9]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})

df['Gender'].value_counts()

Gender
1    360
0    243
Name: count, dtype: int64

In [10]:
df['Age_at_scan_years'].describe()


count    603.000000
mean      63.176418
std       11.985623
min       20.740000
25%       56.125000
50%       63.580000
75%       71.640000
max       88.500000
Name: Age_at_scan_years, dtype: float64

In [11]:
df['IDH1'].value_counts()

IDH1
Wildtype    496
NOS/NEC      94
Mutated      13
Name: count, dtype: int64

In [12]:
df['MGMT'].value_counts()

MGMT
Not Available    313
Unmethylated     159
Methylated       106
Indeterminate     25
Name: count, dtype: int64

In [13]:
df['GTR_over90percent'].value_counts()

GTR_over90percent
Y                 347
N                 204
Not Available      34
Not Applicable     18
Name: count, dtype: int64

In [14]:
df['KPS'].value_counts()


KPS
Not Available    529
90                37
80                18
60                 6
70                 5
100                5
40                 2
30                 1
Name: count, dtype: int64

In [15]:
df['KPS'] = pd.to_numeric(df['KPS'], errors='coerce')
# Define the bins: VERY LITTLE DATA, so keep it in just 3 beans
# Groups: 0-70 (Impaired), 80-100 (Functional), NaN (Unknown)
def bin_kps(score):
    if pd.isna(score):
        return 'Unknown'
    elif score <= 80:
        return 'High_KPS'
    else:
        return 'Low_KPS'
df['KPS'] = df['KPS'].apply(bin_kps)
        

df['IDH1'] = df['IDH1'].replace(['NOS/NEC'], 'Unknown').fillna('Unknown')

# 2. Standardize MGMT (Map Indeterminate and Not Available to 'Unknown')
df['MGMT'] = df['MGMT'].replace(['Not Available', 'Indeterminate'], 'Unknown').fillna('Unknown')

# 3. Standardize GTR (Map Not Available/Applicable to 'Unknown')
df['GTR_over90percent'] = df['GTR_over90percent'].replace(['Not Available', 'Not Applicable'], 'Unknown').fillna('Unknown')

# 4. Create Dummies for these columns
# This will create columns like: MGMT_Methylated, MGMT_Unmethylated, MGMT_Unknown
clinical_cols = ['KPS', 'IDH1', 'MGMT', 'GTR_over90percent']
df = pd.get_dummies(df, columns=clinical_cols, prefix=clinical_cols)

# 5. Convert any remaining boolean dummies to int (0/1)
# This ensures your neural network receives a numeric matrix
for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)
        

In [16]:
df.head()

,ID,Gender,Age_at_scan_years,Survival_from_surgery_days_UPDATED,Survival_Status,Survival_Censor,Time_since_baseline_preop,PsP_TP_score,KPS_High_KPS,KPS_Low_KPS,KPS_Unknown,IDH1_Mutated,IDH1_Unknown,IDH1_Wildtype,MGMT_Methylated,MGMT_Unknown,MGMT_Unmethylated,GTR_over90percent_N,GTR_over90percent_Unknown,GTR_over90percent_Y
0,UPENN-GBM-00001_11,0,52.16,960,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
1,UPENN-GBM-00002_11,0,61.30,291,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
2,UPENN-GBM-00003_11,1,42.82,2838,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1
3,UPENN-GBM-00004_11,1,33.43,623,Deceased,Not Available,0,NaN,0,0,1,0,1,0,0,1,0,0,0,1
4,UPENN-GBM-00005_11,1,53.33,1143,Deceased,Not Available,0,NaN,0,0,1,0,0,1,0,1,0,0,0,1


# Dataloader

In [22]:
all_ids = set(df['ID'].map(lambda x: x.split('_')[0]))
print(f"Total unique IDs in clinical data: {len(all_ids)}")

Total unique IDs in clinical data: 603


In [23]:
from sklearn.model_selection import train_test_split
from maikol_utils.file_utils import save_json

# train_test_split needs an indexable sequence, not a set.
candidate_ids = sorted(all_ids)

# First split off the combined validation + test pool.
train_ids, aux_ids = train_test_split(
    candidate_ids,
    test_size=CONFIG.test_split + CONFIG.val_split,
    random_state=CONFIG.seed,
    shuffle=True,
)

# Then split the remaining  val/test with the requested counts.
val_ids, test_ids = train_test_split(
    aux_ids,
    test_size=CONFIG.val_split / (CONFIG.test_split + CONFIG.val_split),
    random_state=CONFIG.seed,
    shuffle=True,
)

partition_ids = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids,
}

save_json(CONFIG.partition_ids_path, partition_ids)

print(f"Saved partition ids at: {CONFIG.partition_ids_path}")
print({k: len(v) for k, v in partition_ids.items()})
print(f"Total: {sum(len(v) for v in partition_ids.values())}")

Saving output at ../data/partitions_ids.json...
Saved partition ids at: ../data/partitions_ids.json
{'train': 482, 'val': 60, 'test': 61}
Total: 603


In [24]:
from src.data import UPennGBMDataset

data_train = UPennGBMDataset(
    CONFIG,
    partition='train'
)
data_val = UPennGBMDataset(
    CONFIG,
    partition='val'
)
data_test = UPennGBMDataset(
    CONFIG,
    partition='test'
)
print(f"Train set size: {len(data_train)}")
print(f"Val set size: {len(data_val)}")
print(f"Test set size: {len(data_test)}")

Train set size: 482
Val set size: 60
Test set size: 61


In [25]:
train_ids = set(data_train.df['ID'])
val_ids = set(data_val.df['ID'])
test_ids = set(data_test.df['ID'])

all_ids = train_ids.union(val_ids).union(test_ids)
print(f"Total unique IDs across all partitions: {len(all_ids)}")

print(f"Train IDs: {len(train_ids)}")
print(f"Val IDs: {len(val_ids)}")
print(f"Test IDs: {len(test_ids)}")

Total unique IDs across all partitions: 603
Train IDs: 482
Val IDs: 60
Test IDs: 61


# Radiomic

In [11]:
df_rad = pd.read_csv(files[0])

list(df_rad.columns)

['SubjectID',
 'FLAIR_ED_Intensity_CoefficientOfVariation',
 'FLAIR_ED_Intensity_Energy',
 'FLAIR_ED_Intensity_InterQuartileRange',
 'FLAIR_ED_Intensity_Kurtosis',
 'FLAIR_ED_Intensity_Maximum',
 'FLAIR_ED_Intensity_Mean',
 'FLAIR_ED_Intensity_MeanAbsoluteDeviation',
 'FLAIR_ED_Intensity_Median',
 'FLAIR_ED_Intensity_MedianAbsoluteDeviation',
 'FLAIR_ED_Intensity_Minimum',
 'FLAIR_ED_Intensity_Mode',
 'FLAIR_ED_Intensity_NinetiethPercentile',
 'FLAIR_ED_Intensity_QuartileCoefficientOfVariation',
 'FLAIR_ED_Intensity_Range',
 'FLAIR_ED_Intensity_RootMeanSquare',
 'FLAIR_ED_Intensity_Skewness',
 'FLAIR_ED_Intensity_StandardDeviation',
 'FLAIR_ED_Intensity_Sum',
 'FLAIR_ED_Intensity_TenthPercentile',
 'FLAIR_ED_Intensity_Variance',
 'FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-0_Frequency',
 'FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-0_Probability',
 'FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-10_Frequency',
 'FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-10_Probability',
 'FLAIR_ED_Histogram_Bins-16_Bins-1

In [20]:
merged_df = pd.DataFrame()

cols = set()
for f in files:
    df_rad = pd.read_csv(f)
    cols.update(df_rad.columns)
    # merged_df = merged_df.merge(df_rad, on="SubjectID", how="outer")


cols

{'FLAIR_NC_Histogram_Bins-16_Bins-16_Bin-2_Frequency',
 'T2_ED_Histogram_Bins-16_Bins-16_TenthPercentile',
 'T2_ED_Intensity_Maximum',
 'T2_NC_Histogram_Bins-16_Bins-16_Bin-13_Frequency',
 'T1GD_ET_Histogram_Bins-16_Bins-16_Bin-12_Frequency',
 'T2_ET_Histogram_Bins-16_Bins-16_Bin-8_Probability',
 'T1_NC_Volumetric_Pixels',
 'FLAIR_ET_Morphologic_EllipseDiameter_Axis-1',
 'T1GD_NC_Intensity_NinetiethPercentile',
 'T2_ED_Histogram_Bins-16_Bins-16_Range',
 'T2_NC_GLCM_Bins-16_Radius-1_Energy',
 'T2_NC_Histogram_Bins-16_Bins-16_Bin-11_Probability',
 'FLAIR_ED_GLSZM_Bins-16_Radius-1_ZonePercentage',
 'FLAIR_ED_Morphologic_PerimeterOnBorder',
 'FLAIR_NC_GLCM_Bins-16_Radius-1_Correlation',
 'FLAIR_ET_Histogram_Bins-16_Bins-16_Uniformity',
 'T1GD_ET_GLSZM_Bins-16_Radius-1_GreyLevelNonUniformityNormalized',
 'T1GD_ED_GLSZM_Bins-16_Radius-1_LargeZoneLowGreyLevelEmphasis',
 'T2_NC_NGTDM_Busyness',
 'T1_ET_Intensity_MedianAbsoluteDeviation',
 'T1GD_ED_Histogram_Bins-16_Bins-16_NinetiethPercentile'

In [24]:
from pathlib import Path

files = [
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_NC.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ED.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ET.csv",
    "../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_NC.csv",
]

merged_df = None
for f in files:
    path = Path(f)
    print(f"Loading {path}...")
    df_rad = pd.read_csv(path)
    if "SubjectID" not in df_rad.columns:
        print(f"Skipping {path.name}: missing SubjectID")
        continue

    # Prefix feature columns with file stem to avoid name collisions across files.
    feature_cols = [c for c in df_rad.columns if c != "SubjectID"]
    df_rad = df_rad.rename(columns={c: f"{path.stem}__{c}" for c in feature_cols})
    
    if merged_df is None:
        merged_df = df_rad
    else:
        merged_df = merged_df.merge(df_rad, on="SubjectID", how="outer")

if merged_df is None:
    raise ValueError("No valid files with SubjectID were loaded.")

merged_df['Patient'] = merged_df['SubjectID'].str.split('_').str[0]
# 2. Sort by SubjectID so that _11 comes before _21 for the same patient
merged_df = merged_df.sort_values(by='SubjectID')
# 3. Drop duplicates based on the Root SubjectID, keeping the first one found
# This keeps _11 if it exists; otherwise, it keeps _21.
merged_df = merged_df.drop_duplicates(subset='Patient', keep='first')
merged_df['SubjectID'] = merged_df['Patient']

print(f"Merged shape: {merged_df.shape}")
print(f"Unique subjects: {merged_df['SubjectID'].nunique()}")
merged_df.head()

Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_ET.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_FLAIR_NC.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ED.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_ET.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1_NC.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ED.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_ET.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T1GD_NC.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ED.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_ET.csv...
Loading ../data/radiomic/Radiomic_Features_CaPTk_automaticsegm_T2_NC.csv...
Merged shape: (611, 1730)
Unique subjects: 611


/tmp/ipykernel_1817098/2426528060.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df['Patient'] = merged_df['SubjectID'].str.split('_').str[0]


,SubjectID,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_CoefficientOfVariation,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Energy,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_InterQuartileRange,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Kurtosis,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Maximum,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Mean,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_MeanAbsoluteDeviation,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Median,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_MedianAbsoluteDeviation,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Minimum,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Mode,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_NinetiethPercentile,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_QuartileCoefficientOfVariation,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Range,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_RootMeanSquare,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Skewness,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_StandardDeviation,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Sum,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_TenthPercentile,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Intensity_Variance,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-0_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-0_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-10_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-10_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-11_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-11_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-12_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-12_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-13_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-13_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-14_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-14_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-15_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-15_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-1_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-1_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-2_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-2_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-3_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-3_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-4_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-4_Probability,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-16_Bins-16_Bin-5_Frequency,Radiomic_Features_CaPTk_automaticsegm_FLAIR_ED__FLAIR_ED_Histogram_Bins-1

In [40]:
radiomic_ids = set(merged_df['SubjectID'].unique())
len(radiomic_ids)

611

In [41]:
tabular_ids = train_ids.union(val_ids).union(test_ids)
print(f"Total unique IDs in data: {len(tabular_ids)}")

Total unique IDs in data: 603


In [42]:
len(set(tabular_ids) | set(radiomic_ids))

629

In [43]:
with_all_ids = [id for id in tabular_ids if id in radiomic_ids]
len(with_all_ids)

585

In [44]:
from maikol_utils.file_utils import save_json

save_json(CONFIG.tabular_ids_path, list(tabular_ids))
save_json(CONFIG.radiomic_ids_path, list(radiomic_ids))
save_json(CONFIG.with_all_ids_path, list(with_all_ids))

Saving output at ../data/tabular_ids.json...
Saving output at ../data/radiomic_ids.json...
Saving output at ../data/with_all_ids.json...


['UPENN-GBM-00421',
 'UPENN-GBM-00462',
 'UPENN-GBM-00409',
 'UPENN-GBM-00354',
 'UPENN-GBM-00384',
 'UPENN-GBM-00236',
 'UPENN-GBM-00457',
 'UPENN-GBM-00020',
 'UPENN-GBM-00059',
 'UPENN-GBM-00233',
 'UPENN-GBM-00267',
 'UPENN-GBM-00245',
 'UPENN-GBM-00210',
 'UPENN-GBM-00139',
 'UPENN-GBM-00464',
 'UPENN-GBM-00019',
 'UPENN-GBM-00561',
 'UPENN-GBM-00286',
 'UPENN-GBM-00240',
 'UPENN-GBM-00140',
 'UPENN-GBM-00588',
 'UPENN-GBM-00357',
 'UPENN-GBM-00562',
 'UPENN-GBM-00274',
 'UPENN-GBM-00063',
 'UPENN-GBM-00559',
 'UPENN-GBM-00007',
 'UPENN-GBM-00550',
 'UPENN-GBM-00412',
 'UPENN-GBM-00141',
 'UPENN-GBM-00221',
 'UPENN-GBM-00246',
 'UPENN-GBM-00091',
 'UPENN-GBM-00060',
 'UPENN-GBM-00057',
 'UPENN-GBM-00465',
 'UPENN-GBM-00053',
 'UPENN-GBM-00508',
 'UPENN-GBM-00486',
 'UPENN-GBM-00006',
 'UPENN-GBM-00440',
 'UPENN-GBM-00607',
 'UPENN-GBM-00049',
 'UPENN-GBM-00144',
 'UPENN-GBM-00225',
 'UPENN-GBM-00034',
 'UPENN-GBM-00013',
 'UPENN-GBM-00157',
 'UPENN-GBM-00514',
 'UPENN-GBM-00038',


# Random droptout

In [28]:
from maikol_utils.file_utils import load_json

all_ids = load_json(CONFIG.partition_ids_path)

all_ids = all_ids['train'] + all_ids['val'] + all_ids['test']
print(f"Total unique IDs in tabular or radiomic: {len(all_ids)}")

Loading output from ../data/partitions_ids.json...
Total unique IDs in tabular or radiomic: 603


In [29]:
data_train.df.columns

Index(['ID', 'Gender', 'Age_at_scan_years', 'KPS_High', 'KPS_Low', 'KPS_Unk',
       'IDH1_Mutated', 'IDH1_Wildtype', 'IDH1_Unk', 'MGMT_Methylated',
       'MGMT_Unmethylated', 'MGMT_Unk', 'GTR_Y', 'GTR_N', 'GTR_Unk',
       'Survival_Class'],
      dtype='str')

In [41]:
p = 0.1
drop_config = {
    'T1': 0.05,
    'T1GD': 0.05,
    'T2': 0.05,
    'FLAIR': 0.05,
    'RADIOMIC': {
        'ET': (['T1_ET', 'T1GD_ET', 'T2_ET', 'FLAIR_ET'], p),
        'ED': (['T1_ED', 'T1GD_ED', 'T2_ED', 'FLAIR_ED'], p),
        'NC': (['T1_NC', 'T1GD_NC', 'T2_NC', 'FLAIR_NC'], p),
    },
    'TABULAR': {
        'Gender': (['Gender'], p),
        'Age': (['Age_at_scan_years'], p),
        'KPS': (['KPS_High', 'KPS_Low', 'KPS_Unk'], p),
        'IDH1': (['IDH1_Mutated', 'IDH1_Wildtype', 'IDH1_Unk'], p),
        'MGMT': (['MGMT_Methylated', 'MGMT_Unmethylated', 'MGMT_Unk'], p),
        'GTR': (['GTR_Y', 'GTR_N', 'GTR_Unk'], p),
    }
}

total_prob = 1.0
for modality, drop_rate in drop_config.items():
    if isinstance(drop_rate, dict):
        for submod, (cols, sub_drop_rate) in drop_rate.items():
            total_prob *= (1 - sub_drop_rate)
    else:
        total_prob *= (1 - drop_rate)
    
print(f"Have all features probability: {total_prob:.4f}")

Have all features probability: 0.3156


In [42]:
import random

# Optional seed for reproducibility across runs.
rng = random.Random(42)

def sample_drop_state(config, rng_obj):
    state = {}
    for key, value in config.items():
        if isinstance(value, dict):
            state[key] = {
                subkey: (rng_obj.random() < prob[1])
                for subkey, prob in value.items()
            }
        else:
            state[key] = rng_obj.random() < value
    return state

# One boolean dictionary per ID with the same shape as drop_config.
drop_state_by_id = {id_: sample_drop_state(drop_config, rng) for id_ in all_ids}

print(f"Generated dropout state for {len(drop_state_by_id)} IDs")

Generated dropout state for 603 IDs


In [44]:
dropout_data ={
    'reference': drop_config,
    "ids": drop_state_by_id
}

save_json(CONFIG.dropout_data_path, dropout_data)

Saving output at ../data/dropout_data.json...


{'reference': {'T1': 0.05,
  'T1GD': 0.05,
  'T2': 0.05,
  'FLAIR': 0.05,
  'RADIOMIC': {'ET': (['T1_ET', 'T1GD_ET', 'T2_ET', 'FLAIR_ET'], 0.1),
   'ED': (['T1_ED', 'T1GD_ED', 'T2_ED', 'FLAIR_ED'], 0.1),
   'NC': (['T1_NC', 'T1GD_NC', 'T2_NC', 'FLAIR_NC'], 0.1)},
  'TABULAR': {'Gender': (['Gender'], 0.1),
   'Age': (['Age_at_scan_years'], 0.1),
   'KPS': (['KPS_High', 'KPS_Low', 'KPS_Unk'], 0.1),
   'IDH1': (['IDH1_Mutated', 'IDH1_Wildtype', 'IDH1_Unk'], 0.1),
   'MGMT': (['MGMT_Methylated', 'MGMT_Unmethylated', 'MGMT_Unk'], 0.1),
   'GTR': (['GTR_Y', 'GTR_N', 'GTR_Unk'], 0.1)}},
 'ids': {'UPENN-GBM-00010': {'T1': False,
   'T1GD': True,
   'T2': False,
   'FLAIR': False,
   'RADIOMIC': {'ET': False, 'ED': False, 'NC': False},
   'TABULAR': {'Gender': True,
    'Age': False,
    'KPS': True,
    'IDH1': False,
    'MGMT': False,
    'GTR': True}},
  'UPENN-GBM-00445': {'T1': False,
   'T1GD': False,
   'T2': False,
   'FLAIR': False,
   'RADIOMIC': {'ET': False, 'ED': False, 'NC': True